In [2]:
//4
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv(r"C:\Users\Tanuja\Downloads\california_housing.csv")

df.columns = df.columns.astype(str).str.strip()
df = df.loc[:, ~df.columns.str.lower().str.startswith("unnamed")]

target_col = None

for col in df.columns:
    if col.lower().replace("_", "").replace(" ", "") in [
        "medhouseval", "medianhousevalue", "target"
    ]:
        target_col = col
        break

if target_col is None:
    raise ValueError("Target column not found. Check dataset columns.")

X = df.drop(columns=[target_col]).select_dtypes(include="number")
y = pd.to_numeric(df[target_col], errors="coerce")

data = pd.concat([X, y.rename("target")], axis=1).dropna()

X = data.drop(columns=["target"])
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

results = []

for degree in [1, 2, 3]:

    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression())
    ])

    start = time.perf_counter()
    model.fit(X_train, y_train)
    training_time = time.perf_counter() - start

    start = time.perf_counter()
    y_test_pred = model.predict(X_test)
    prediction_time = time.perf_counter() - start

    y_train_pred = model.predict(X_train)

    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    results.append({
        "Degree": degree,
        "Features": model.named_steps["poly"].n_output_features_,
        "Training MSE": train_mse,
        "Testing MSE": test_mse,
        "Training R2": train_r2,
        "Testing R2": test_r2,
        "Training Time": training_time,
        "Prediction Time": prediction_time
    })

results_df = pd.DataFrame(results)

print("Polynomial Regression Results:")
print(results_df.to_string(index=False))


Polynomial Regression Results:
 Degree  Features  Training MSE  Testing MSE  Training R2  Testing R2  Training Time  Prediction Time
      1         8      0.517933     0.555892     0.612551    0.575788       0.026193         0.005010
      2        44      0.420727     0.464302     0.685268    0.645682       0.071727         0.007272
      3       164      0.342026    25.407137     0.744142  -18.388708       0.210844         0.019229


In [ ]:
%pip install pandas numpy scikit-learn
# 1. AIM

To demonstrate the use of Polynomial Regression with varying polynomial degrees on a suitable real-world dataset and observe the effect of model complexity on prediction accuracy in terms of the Bias–Variance Trade-off.

# 2. DATASET USED

**Dataset Name:** California Housing Dataset

**Dataset Path:** `C:\Users\Tanuja\Downloads\california_housing.csv`

The California Housing dataset contains housing-related information about different districts in California. It is used to predict the median house value based on various housing and demographic attributes.

The standard dataset contains 20,640 records, 8 numerical input features and one continuous target variable.

### Features of the Dataset

| Feature | Description |
|---|---|
| MedInc | Median income in the block group |
| HouseAge | Median house age |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Population of the block group |
| AveOccup | Average number of household members |
| Latitude | Geographical latitude |
| Longitude | Geographical longitude |
| MedHouseVal | Median house value (target) |

**Why is this dataset used?**

1. It represents a real-world housing problem.
2. It contains multiple numerical input features.
3. It has a continuous numerical target variable.
4. It is suitable for applying Polynomial Regression.
5. It helps demonstrate how different polynomial degrees affect prediction performance.

# 3. THEORY

## A. Regression

Regression is a supervised machine learning technique used to predict continuous numerical values by identifying relationships between input variables and an output variable.

Examples include house price prediction, salary prediction and sales forecasting.

## B. Polynomial Regression

Polynomial Regression is a regression technique used to model nonlinear relationships between independent and dependent variables by introducing polynomial terms of the input features.

It extends Linear Regression by adding squared, cubed and higher-degree terms of the original features.

Polynomial Regression can fit curved relationships between input and output variables.

### Mathematical Equation

For one independent variable:

Y = β₀ + β₁X + β₂X² + β₃X³ + ... + βₙXⁿ + ε

Where:

- Y = Dependent variable.
- X = Independent variable.
- β₀ = Intercept.
- β₁, β₂, ... βₙ = Regression coefficients.
- n = Polynomial degree.
- ε = Error term.

## C. Polynomial Degree

Polynomial degree represents the highest power of the input variable included in the regression equation.

Different degrees produce models with different levels of complexity.

**1. Degree 1 – Linear Regression**

Y = β₀ + β₁X

It produces a straight-line relationship between input and output.

**2. Degree 2 – Quadratic Regression**

Y = β₀ + β₁X + β₂X²

It introduces a squared term and can represent curved relationships.

**3. Degree 3 – Cubic Regression**

Y = β₀ + β₁X + β₂X² + β₃X³

It introduces a cubic term and can represent more complex curved relationships.

In this practical, degrees 1, 2 and 3 are compared to observe the effect of model complexity.

## D. Working of Polynomial Regression

Polynomial Regression works through the following steps:

1. Load the dataset.
2. Identify independent and dependent variables.
3. Divide the dataset into training and testing sets.
4. Generate polynomial features for the selected degree.
5. Standardize the generated features.
6. Train the Linear Regression model.
7. Predict values using testing data.
8. Calculate regression evaluation metrics.
9. Repeat the process for different polynomial degrees.
10. Compare the results.

## E. PolynomialFeatures

PolynomialFeatures is a preprocessing technique provided by Scikit-learn.

It generates polynomial combinations of the original input features up to a specified degree.

For example, if the original features are X₁ and X₂:

For degree 1:

X₁, X₂

For degree 2:

X₁, X₂, X₁², X₁X₂, X₂²

For degree 3:

X₁, X₂, X₁², X₁X₂, X₂², X₁³, X₁²X₂, X₁X₂², X₂³

Thus, increasing the polynomial degree generates more features and increases model complexity.

## F. Bias–Variance Trade-off

Bias–Variance Trade-off describes the balance between two types of prediction errors: bias and variance.

It helps us understand how model complexity affects prediction performance.

**1. Bias**

Bias is the error caused by overly simple assumptions in a machine learning model.

A model with high bias may fail to capture important patterns in the data.

This generally leads to underfitting.

**2. Variance**

Variance refers to how much a model's predictions change when it is trained on different training datasets.

A model with high variance may learn small variations and noise in the training data.

This generally leads to overfitting.

**3. Underfitting**

Underfitting occurs when the model is too simple to learn the actual relationship between input and output variables.

Characteristics:

- High training error.
- High testing error.
- High bias.
- Model fails to capture important patterns.

**4. Overfitting**

Overfitting occurs when the model learns the training data too closely, including noise and unnecessary patterns.

Characteristics:

- Very low training error.
- Higher testing error.
- High variance.
- Poor generalization to unseen data.

**5. Good Fit**

A good-fit model captures important patterns without learning unnecessary noise.

Characteristics:

- Low training error.
- Low testing error.
- Reasonable bias and variance.
- Good generalization.

## G. Influence of Polynomial Degree on Model Complexity

The polynomial degree controls the complexity of the model.

- Degree 1 may underfit when the actual relationship is nonlinear.
- Degree 2 can capture quadratic relationships.
- Degree 3 can capture more complex cubic relationships.
- Very high degrees may cause overfitting.

Increasing polynomial degree does not always improve testing accuracy.

The best degree is selected by observing testing performance and the difference between training and testing errors.

## H. Regression Evaluation Metrics

The following metrics are used to evaluate Polynomial Regression.

### 1. Mean Squared Error (MSE)

MSE calculates the average squared difference between actual and predicted values.

Formula:

MSE = (1/n) × Σ(Yᵢ - Ŷᵢ)²

A lower MSE indicates smaller squared prediction errors.

### 2. Root Mean Squared Error (RMSE)

RMSE is the square root of MSE.

Formula:

RMSE = √MSE

It measures prediction error in the same unit as the target variable.

### 3. R² Score

R² measures how much variation in the dependent variable is explained by the regression model.

Formula:

R² = 1 - (SSres / SStot)

Where:

- SSres = Sum of squared residuals.
- SStot = Total sum of squares.

A higher R² generally indicates better model fit. It can also be negative when the model performs worse than a constant mean-based prediction.

## I. Role of StandardScaler

StandardScaler standardizes numerical features to approximately zero mean and unit standard deviation.

Formula:

Z = (X - μ) / σ

It is used after generating polynomial features in this practical to bring the expanded features onto comparable scales.

## J. Advantages of Polynomial Regression

1. Models nonlinear relationships.
2. Captures curved patterns in data.
3. Extends Linear Regression using polynomial features.
4. Allows comparison of different model complexities.
5. Can improve prediction performance when the relationship is nonlinear.

## K. Limitations of Polynomial Regression

1. Higher degrees generate more features.
2. High-degree models may overfit.
3. Training and prediction can become computationally expensive.
4. Polynomial features can become highly correlated.
5. The model may perform poorly on unseen data if its complexity is too high.

# 4. ALGORITHM

1. Start.
2. Import Pandas, NumPy and required Scikit-learn libraries.
3. Load the California Housing dataset using read_csv().
4. Identify the target column.
5. Select numerical independent variables.
6. Remove rows containing missing input or target values.
7. Separate independent variables X and dependent variable y.
8. Divide the dataset into training and testing sets.
9. Initialize an empty list to store the results.
10. Select polynomial degrees 1, 2 and 3.
11. Generate polynomial features using PolynomialFeatures().
12. Standardize the generated features using StandardScaler().
13. Train the Linear Regression model using fit().
14. Predict training and testing values using predict().
15. Calculate training MSE and testing MSE.
16. Calculate training R² and testing R².
17. Measure training and prediction time.
18. Store the results for each polynomial degree.
19. Display the comparison of all polynomial degrees.
20. Stop.

# 5. DETAILED EXPLANATION OF EACH LINE OF CODE

## A. Importing Libraries

**1. import pandas as pd**

Imports Pandas for loading and manipulating datasets.

**2. import numpy as np**

Imports NumPy for numerical calculations.

**3. import time**

Imports the time module to measure training and prediction time.

**4. from sklearn.model_selection import train_test_split**

Imports train_test_split() to divide data into training and testing sets.

**5. from sklearn.preprocessing import PolynomialFeatures, StandardScaler**

Imports two preprocessing classes:

- PolynomialFeatures generates polynomial and interaction terms.
- StandardScaler standardizes the generated features.

**6. from sklearn.pipeline import Pipeline**

Imports Pipeline to combine polynomial transformation, scaling and regression into a single workflow.

**7. from sklearn.linear_model import LinearRegression**

Imports LinearRegression to train the regression model using generated polynomial features.

**8. from sklearn.metrics import mean_squared_error, r2_score**

Imports two evaluation functions:

- mean_squared_error() calculates MSE.
- r2_score() calculates R².

## B. Loading the Dataset

`df = pd.read_csv(r"C:\Users\Tanuja\Downloads\california_housing.csv")`

- read_csv() loads the California Housing dataset.
- df stores the dataset in a Pandas DataFrame.
- The specified path identifies the dataset location.

## C. Cleaning Column Names

`df.columns = df.columns.astype(str).str.strip()`

Converts column names into strings and removes unnecessary spaces.

`df = df.loc[:, ~df.columns.str.lower().str.startswith("unnamed")]`

Removes unwanted unnamed columns that may have been generated during CSV export.

## D. Identifying Target Variable

`target_col = None`

Initially sets the target variable as empty.

`for col in df.columns:`

Checks every column in the dataset.

`col.lower().replace("_", "").replace(" ", "")`

Converts the column name to lowercase and removes underscores and spaces for comparison.

`if target_col is None:`

Checks whether a suitable target column was identified.

`raise ValueError(...)`

Displays an error if the target column cannot be found.

## E. Selecting Independent Variables

`X = df.drop(columns=[target_col]).select_dtypes(include="number")`

- drop() removes the target column.
- select_dtypes() selects numerical columns.
- X stores the independent variables.

## F. Selecting Dependent Variable

`y = pd.to_numeric(df[target_col], errors="coerce")`

Converts the target column into numerical form.

`errors="coerce"` converts invalid values into missing values.

## G. Removing Missing Values

`data = pd.concat([X, y.rename("target")], axis=1).dropna()`

Combines input and target columns and removes rows containing missing values.

`X = data.drop(columns=["target"])`

Extracts the input features again.

`y = data["target"]`

Extracts the dependent variable.

## H. Splitting the Dataset

`train_test_split(X, y, test_size=0.2, random_state=42)`

Divides the dataset into:

- 80% training data.
- 20% testing data.

random_state=42 ensures that the same split is obtained whenever the program is executed with the same dataset.

## I. Initializing Results

`results = []`

Creates an empty Python list to store the performance results for different polynomial degrees.

## J. Selecting Polynomial Degrees

`for degree in [1, 2, 3]:`

Executes the same process three times using polynomial degrees 1, 2 and 3.

This helps compare model complexity.

## K. Creating the Polynomial Regression Model

`PolynomialFeatures(degree=degree, include_bias=False)`

Generates polynomial features according to the selected degree.

- degree specifies the maximum polynomial degree.
- include_bias=False prevents adding an additional constant column because LinearRegression already includes an intercept by default.

`StandardScaler()`

Standardizes the generated polynomial features.

`LinearRegression()`

Creates the regression model.

`Pipeline([...])`

Combines all three operations into one workflow:

1. Polynomial feature generation.
2. Feature scaling.
3. Linear Regression.

## L. Measuring Training Time

`start = time.perf_counter()`

Records the starting time.

`model.fit(X_train, y_train)`

Fits the complete pipeline using the training dataset.

It generates polynomial features, scales them and trains the Linear Regression model.

`training_time = time.perf_counter() - start`

Calculates the total training and preprocessing time.

## M. Making Predictions

`y_test_pred = model.predict(X_test)`

Predicts target values using the testing data.

`prediction_time = time.perf_counter() - start`

Measures the time required for prediction, including the transformations in the pipeline.

`y_train_pred = model.predict(X_train)`

Predicts values for the training dataset to evaluate training performance.

## N. Calculating Training and Testing Error

`mean_squared_error(y_train, y_train_pred)`

Calculates the training Mean Squared Error.

`mean_squared_error(y_test, y_test_pred)`

Calculates the testing Mean Squared Error.

These measurements help observe whether a model is underfitting or overfitting.

## O. Calculating R² Score

`r2_score(y_train, y_train_pred)`

Calculates the R² score for training data.

`r2_score(y_test, y_test_pred)`

Calculates the R² score for testing data.

It measures the amount of variation explained by the model.

## P. Storing Results

`results.append({...})`

Stores the following information for every polynomial degree:

- Polynomial degree.
- Number of generated features.
- Training MSE.
- Testing MSE.
- Training R².
- Testing R².
- Training time.
- Prediction time.

## Q. Displaying Results

`results_df = pd.DataFrame(results)`

Converts the result list into a Pandas DataFrame.

`results_df.to_string(index=False)`

Displays the complete comparison table without row indices.

# 6. EXPECTED OUTPUT

For the standard California Housing dataset with eight numerical input features:

| Polynomial degree | Generated features |
|---|---:|
| Degree 1 | 8 |
| Degree 2 | 44 |
| Degree 3 | 164 |

The exact feature count is based on using all eight original numerical input features and `include_bias=False`.

The output comparison table contains:

1. Degree.
2. Number of generated features.
3. Training MSE.
4. Testing MSE.
5. Training R².
6. Testing R².
7. Training time.
8. Prediction time.

The metric values and execution times are calculated during program execution. They should not be assumed or copied as fixed values.

# 7. ORAL EXAMINATION – VIVA QUESTIONS AND ANSWERS

## Basic Questions

**Q1. What is Polynomial Regression?**

Polynomial Regression is a regression technique that models nonlinear relationships by introducing polynomial terms of input variables.

**Q2. Why is Polynomial Regression used?**

It is used when the relationship between independent and dependent variables is nonlinear and cannot be represented adequately by a straight line.

**Q3. What is polynomial degree?**

Polynomial degree represents the highest power of the input variable or variables included in the polynomial model.

**Q4. What is degree 1 Polynomial Regression?**

Degree 1 represents an ordinary linear regression model without additional polynomial terms.

**Q5. What is degree 2 Polynomial Regression?**

It includes squared terms and interaction terms up to degree 2.

**Q6. What is degree 3 Polynomial Regression?**

It includes cubic terms and interaction terms up to degree 3.

**Q7. What is PolynomialFeatures?**

It is a Scikit-learn preprocessing class that generates polynomial combinations of input features.

**Q8. What is an interaction term?**

An interaction term represents the combined effect of two or more input variables. For example, X₁X₂.

**Q9. What is model complexity?**

Model complexity refers to how flexible a model is in learning patterns from the dataset.

**Q10. Which dataset is used in this practical?**

California Housing Dataset.

## Bias–Variance Questions

**Q11. What is Bias?**

Bias is the error caused by overly simple assumptions in a machine learning model.

**Q12. What is Variance?**

Variance measures how much a model's predictions change when it is trained on different datasets.

**Q13. What is the Bias–Variance Trade-off?**

It is the balance between bias and variance to achieve good prediction performance on unseen data.

**Q14. What is Underfitting?**

Underfitting occurs when a model is too simple to learn the important patterns in the data.

**Q15. What is Overfitting?**

Overfitting occurs when a model learns the training data too closely, including noise, and performs poorly on unseen data.

**Q16. How does polynomial degree affect model complexity?**

Increasing the polynomial degree generates more features and allows the model to learn more complex relationships.

**Q17. Which degree is more likely to underfit?**

A very low degree may underfit when the underlying relationship is more complex.

**Q18. Which degree is more likely to overfit?**

A very high degree may overfit by learning noise and unnecessary patterns.

**Q19. How can you identify overfitting?**

When training error is very low but testing error is considerably higher, the model may be overfitting.

**Q20. How can you identify underfitting?**

When both training and testing errors are high, the model may be underfitting.

# 8. DIFFICULT EXTERNAL EXAMINER QUESTIONS AND ANSWERS

**Q21. Why is Polynomial Regression called Linear Regression internally?**

Because it is linear in its coefficients. Although polynomial features represent curved relationships, the regression equation is linear in the coefficients being estimated.

**Q22. Is Polynomial Regression a linear or nonlinear model?**

It can model nonlinear relationships in the input variables, but it is linear in its regression coefficients.

**Q23. What is the difference between Linear Regression and Polynomial Regression?**

Linear Regression uses original input features, whereas Polynomial Regression expands the input features using polynomial powers and interaction terms.

**Q24. Does increasing polynomial degree always increase accuracy?**

No. Increasing degree can improve training fit but may reduce testing performance due to overfitting.

**Q25. What is the purpose of include_bias=False?**

It prevents PolynomialFeatures from generating an additional column of ones because LinearRegression already includes an intercept by default.

**Q26. Why is StandardScaler applied after PolynomialFeatures in this practical?**

Polynomial expansion creates features with different numerical magnitudes. StandardScaler brings these generated features onto a comparable scale.

**Q27. Why do we evaluate both training and testing performance?**

Training performance shows how well the model fits the training data, while testing performance measures its generalization to unseen data.

**Q28. What is the difference between MSE and RMSE?**

MSE calculates the average squared prediction error. RMSE is the square root of MSE and is expressed in the same unit as the target variable.

**Q29. Can R² be negative?**

Yes. A negative R² indicates that the model performs worse than a constant predictor using the mean of the actual target values.

**Q30. What is the main disadvantage of higher polynomial degrees?**

They generate many features, increase computational requirements and can cause overfitting.

**Q31. What is multicollinearity in Polynomial Regression?**

It occurs when generated polynomial features are highly correlated with each other, which can make regression coefficients unstable.

**Q32. What is the purpose of Pipeline?**

Pipeline combines preprocessing steps and the regression model into a single workflow. It also ensures that transformations are fitted on training data rather than the testing data.

**Q33. Why is random_state=42 used?**

It makes the training and testing split reproducible.

**Q34. How can overfitting be reduced in Polynomial Regression?**

By selecting an appropriate polynomial degree, using cross-validation, increasing training data or applying regularization techniques such as Ridge Regression.

**Q35. Which polynomial degree is the best?**

The best degree is the one that provides good testing performance and generalization without unnecessary model complexity. It must be selected based on the experimental results.

# 9. DATASET-RELATED QUESTIONS

**Q1. What is the California Housing dataset?**

It is a real-world dataset containing housing and demographic information about districts in California.

**Q2. How many records are present in the standard dataset?**

20,640 records.

**Q3. How many numerical input features are present?**

Eight numerical features.

**Q4. What is the target variable?**

MedHouseVal, which represents median house value.

**Q5. What does MedInc represent?**

Median income in the block group.

**Q6. What does HouseAge represent?**

Median house age in the block group.

**Q7. What does AveRooms represent?**

Average number of rooms per household.

**Q8. What does AveBedrms represent?**

Average number of bedrooms per household.

**Q9. What does Population represent?**

Population of the block group.

**Q10. Why is this dataset suitable for Polynomial Regression?**

It contains multiple numerical features and a continuous target variable, allowing us to study polynomial feature expansion and compare different model complexities.

# 10. PRACTICAL EXAMINATION – CODING QUESTIONS

**Q1. Which function loads the dataset?**

`pd.read_csv()`

**Q2. Which function splits the dataset?**

`train_test_split()`

**Q3. Which function generates polynomial features?**

`PolynomialFeatures()`

**Q4. How do you generate degree 2 polynomial features?**

`PolynomialFeatures(degree=2)`

**Q5. How do you generate degree 3 polynomial features?**

`PolynomialFeatures(degree=3)`

**Q6. How do you standardize features?**

`StandardScaler()`

**Q7. Which algorithm is used for regression?**

`LinearRegression()`

**Q8. How do you train the regression model?**

`model.fit(X_train, y_train)`

**Q9. How do you predict output?**

`model.predict(X_test)`

**Q10. How do you calculate MSE?**

`mean_squared_error(y_test, y_pred)`

**Q11. How do you calculate R²?**

`r2_score(y_test, y_pred)`

**Q12. How do you calculate the number of generated polynomial features?**

`model.named_steps["poly"].n_output_features_`

**Q13. How do you measure training time?**

`time.perf_counter()`

# 11. IMPORTANT FORMULAS

**1. Polynomial Regression Equation**

Y = β₀ + β₁X + β₂X² + ... + βₙXⁿ + ε

**2. Mean Squared Error**

MSE = (1/n) × Σ(Yᵢ - Ŷᵢ)²

**3. Root Mean Squared Error**

RMSE = √MSE

**4. R² Score**

R² = 1 - (SSres / SStot)

**5. Standardization**

Z = (X - μ) / σ

# 12. IMPORTANT POINTS FOR VIVA

1. Polynomial Regression models nonlinear relationships using polynomial features.
2. PolynomialFeatures generates powers and interactions of original features.
3. Degree 1 represents linear regression.
4. Degree 2 includes quadratic terms.
5. Degree 3 includes cubic terms.
6. Increasing polynomial degree increases the number of generated features.
7. High model complexity can cause overfitting.
8. Low model complexity can cause underfitting.
9. Bias–Variance Trade-off helps select an appropriate model complexity.
10. Training and testing errors are compared to observe generalization.